In [1]:
!pip install langchain langchain-community langchain-openai faiss-cpu tokenizers


  Using cached langchain-1.4.6-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
Using cached langchain-1.4.6-py3-none-any.whl (170 kB)
Using cached langchain_community-0.4.2-py3-none-any.whl (2.4 MB)

   ---------------------------------------- 0/2 [langchain-community]
   ---------------------------------------- 0/2 [langchain-community]
   ---------------------------------------- 0/2 [langchain-community]
   ---------------------------------------- 0/2 [langchain-community]
   ---------------------------------------- 0/2 [langchain-community]
   ---------------------------------------- 0/2 [langchain-community]
   ---------------------------------------- 0/2 [langchain-community]
   ---------------------------------------- 0/2 [langchain-community]
   ---------------------------------------- 0/2 [langchain-community]
   ---------------------------------------- 0/2 [langchain-community]
   ----------------------------------

In [3]:
# ==========================================================
# ЧАСТЬ 2: СОВРЕМЕННЫЙ ИИ-АГЕНТ НА LANGCHAIN И FAISS (LCEL)
# ==========================================================

import numpy as np
import pandas as pd
from langchain_community.embeddings import FakeEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. БАЗА ЗНАНИЙ ТЕХПОДДЕРЖКИ
knowledge_base = [
    "Ошибка 403 означает проблемы с авторизацией. Для решения нужно зайти в личный кабинет и обновить паспортные данные.",
    "Если не работает интернет и горит красная лампочка LOS, значит поврежден оптический кабель. Требуется вызов мастера.",
    "Роутер 'SkyNet-2000' перезагружается зажатием кнопки Reset на 10 секунд. Стандартный пароль от Wi-Fi указан на наклейке снизу (admin123).",
    "Смена тарифного плана происходит 1-го числа каждого месяца через мобильное приложение. Баланс должен быть положительным."
]

print("=== 1. БАЗА ЗНАНИЙ УСПЕШНО ЗАГРУЖЕНА ===")

# 2. СОЗДАНИЕ ВЕКТОРНОГО ИНДЕКСА В FAISS
# Используем фейковые эмбеддинги, чтобы код работал локально без платных API-ключей
embeddings = FakeEmbeddings(size=1536)
vector_store = FAISS.from_texts(knowledge_base, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 1}) # Настраиваем поиск 1 самого похожего документа
print("=== 2. СЕМАНИТИЧЕСКИЙ ИНДЕКС В FAISS УСПЕШНО СОЗДАН ===")

# 3. НАСТРОЙКА СОВРЕМЕННОЙ ЦЕПОЧКИ LANGCHAIN (LCEL)
prompt_template = """Вы являетесь экспертом техподдержки провайдера SkyNet. 
Используйте только следующие фрагменты контекста для ответа на вопрос. 

Контекст: {context}
Вопрос клиента: {question}

Ответ инженера техподдержки:"""

PROMPT = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

# Функция для красивого форматирования найденных документов в текстовую строку
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Строим современную цепочку LangChain (агент, который связывает поиск в БД и промпт)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | PROMPT
    | StrOutputParser()
)

# 4. ТЕСТИРОВАНИЕ СЕМАНТИЧЕСКОГО ПОИСКА ИИ
user_question = "У меня роутер SkyNet-2000 заглючил, как скинуть настройки и какой там пароль?"
print(f"\nВопрос клиента: '{user_question}'")

# Запускаем семантический поиск через FAISS
found_docs = retriever.invoke(user_question)
print("\n=== ИИ нашел в базе знаний следующий документ: ===")
print(found_docs[0].page_content)


=== 1. БАЗА ЗНАНИЙ УСПЕШНО ЗАГРУЖЕНА ===
=== 2. СЕМАНИТИЧЕСКИЙ ИНДЕКС В FAISS УСПЕШНО СОЗДАН ===

Вопрос клиента: 'У меня роутер SkyNet-2000 заглючил, как скинуть настройки и какой там пароль?'

=== ИИ нашел в базе знаний следующий документ: ===
Роутер 'SkyNet-2000' перезагружается зажатием кнопки Reset на 10 секунд. Стандартный пароль от Wi-Fi указан на наклейке снизу (admin123).
